# Tiny Dreamer Highway — Screening Run

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook is designed to reduce wasted H100 time. Instead of launching one long experiment and hoping it works, it runs a **staged screening schedule** with:

- a **safer config** than the aggressive AMP profile
- **short checkpoints** at multiple cycle counts
- a **quick real-environment demo** after each stage
- optional **auto-stop** if the policy still looks bad

## Strategy

The previous 12-hour H100 run failed catastrophically (avg_reward=1.71, all episodes crash). A full codebase audit against an open-source DreamerV1 reference implementation identified **four critical bugs** (now fixed) and a **six-step integration plan** (now implemented):

### Bugs fixed (commit 18d77cb)
1. **CRITICAL — Sequence training wired in**: World model was trained on single transitions; RSSM GRU started from zeros every time. Now uses `sample_sequences()` + `train_sequence_world_model_step()`.
2. **CRITICAL — Action ordering fixed**: Agent selected actions *before* encoding observations. Now observation encoded first → posterior → action from posterior features.
3. **HIGH — Stochastic actor**: Actor was purely deterministic (`nn.Tanh()`). Now uses tanh-normal distribution with reparametrised sampling.
4. **Resolved — Reward predictor**: Internal convention is consistent (both training and imagination apply predictor after action-driven GRU transition).

### DreamerV1 reference integration (commit 9f1f8c2)
5. **CRITICAL — TD(λ) returns**: Formula used current-state values instead of next-state values, causing value hallucination (imagined_value_mean=38.6 vs avg_reward=1.71). Fixed with `next_values = cat(values[1:], bootstrap)`.
6. **HIGH — Actor exploration**: Added `init_std=5.0`, `mean_scale=5.0`, `min_std=1e-4` with proper `TransformedDistribution(Normal, TanhTransform)` Jacobian correction.
7. **HIGH — Model capacity**: All dims aligned to DreamerV1 reference (embedding=1024, det=200, stoch=30, multi-layer networks). Configurable via `ModelConfig`.
8. **MEDIUM — WM posterior passthrough**: Behavior learning now receives posteriors directly from WM training, eliminating redundant seed re-encoding.
9. **MEDIUM — Kaiming weight init**: `apply_kaiming_init()` applied to all networks.
10. **MEDIUM — Sequence length/horizon**: Increased to 32 and 15 respectively.

This notebook uses a conservative screening config and checks **real-environment reward** at staged checkpoints before committing to a long run.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/CSC_580_Final_Project


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet
python -m pip install flashoptim --quiet || echo "flashoptim unavailable, using standard AdamW"

Already up to date.


From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD


In [3]:
import json
import sys
from pprint import pprint

import torch
import yaml

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.evaluation import record_demo_videos
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'examples' / 'h100_screening_experiment.yaml'
config = load_experiment_config(CONFIG_PATH)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'

print('Loaded config from:', CONFIG_PATH)
print('GPU:', gpu_name)
if 'H100' not in gpu_name:
    print('Warning: this notebook is intended for H100-class runtimes.')

Loaded config from: /content/CSC_580_Final_Project/examples/h100_screening_experiment.yaml
GPU: NVIDIA H100 80GB HBM3


In [4]:
RUN_NAME = 'h100_screen_bugfix_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

config_snapshot_path = RUN_ARTIFACT_ROOT / 'screening_config.yaml'
config_snapshot_path.write_text(
    yaml.safe_dump(config.model_dump(mode='python'), sort_keys=False),
    encoding='utf-8',
)

print('Run name:', RUN_NAME)
print('Saved config snapshot to:', config_snapshot_path)
pprint({
    'max_episode_steps': config.env.max_episode_steps,
    'batch_size': config.training.batch_size,
    'imagination_horizon': config.training.imagination_horizon,
    'world_model_lr': config.training.world_model_lr,
    'actor_lr': config.training.actor_lr,
    'critic_lr': config.training.critic_lr,
    'world_model_updates_per_cycle': config.training.world_model_updates_per_cycle,
    'behavior_updates_per_cycle': config.training.behavior_updates_per_cycle,
    'policy_steps': config.training.policy_steps,
    'sequence_length': config.replay.sequence_length,
})

Run name: h100_screen_run_001
Saved config snapshot to: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/training_runs/h100_screen_run_001/screening_config.yaml
{'actor_lr': 3e-05,
 'batch_size': 128,
 'behavior_updates_per_cycle': 4,
 'critic_lr': 5e-05,
 'imagination_horizon': 8,
 'max_episode_steps': 60,
 'policy_steps': 32,
 'world_model_lr': 0.0003,
 'world_model_updates_per_cycle': 24}


In [5]:
EVAL_CYCLE_MARKS = [25, 50, 100, 150, 250, 400, 800]
DEMO_EPISODES_BY_STAGE = {25: 2, 50: 2, 100: 3, 150: 4, 250: 4, 400: 4, 800: 6}
MIN_AVG_REWARD_BY_STAGE = {25: -1.0, 50: 0.0, 100: 1.5, 150: 3.0, 250: 5.0, 400: 7.0, 800: 10.0}
MIN_AVG_STEPS_BY_STAGE = {25: 4.0, 50: 6.0, 100: 8.0, 150: 10.0, 250: 12.0, 400: 15.0, 800: 18.0}

STAGES = [
    {
        'label': f'screen_{cycles:04d}',
        'cycles': cycles,
        'demo_episodes': DEMO_EPISODES_BY_STAGE[cycles],
        'min_avg_reward': MIN_AVG_REWARD_BY_STAGE[cycles],
        'min_avg_steps': MIN_AVG_STEPS_BY_STAGE[cycles],
    }
    for cycles in EVAL_CYCLE_MARKS
]

AUTO_STOP = True
DEMO_MAX_STEPS = 200

for stage in STAGES:
    print(stage)

{'label': 'screen_0025', 'cycles': 25, 'demo_episodes': 2, 'min_avg_reward': -1.0, 'min_avg_steps': 4.0}
{'label': 'screen_0050', 'cycles': 50, 'demo_episodes': 2, 'min_avg_reward': 0.0, 'min_avg_steps': 6.0}
{'label': 'screen_0100', 'cycles': 100, 'demo_episodes': 3, 'min_avg_reward': 1.5, 'min_avg_steps': 8.0}
{'label': 'screen_0150', 'cycles': 150, 'demo_episodes': 4, 'min_avg_reward': 3.0, 'min_avg_steps': 10.0}
{'label': 'screen_0250', 'cycles': 250, 'demo_episodes': 4, 'min_avg_reward': 5.0, 'min_avg_steps': 12.0}
{'label': 'screen_0400', 'cycles': 400, 'demo_episodes': 4, 'min_avg_reward': 7.0, 'min_avg_steps': 15.0}
{'label': 'screen_0800', 'cycles': 800, 'demo_episodes': 6, 'min_avg_reward': 10.0, 'min_avg_steps': 18.0}


In [6]:
def summarize_demo(bundle):
    rewards = [result.total_reward for result in bundle.results]
    steps = [result.steps for result in bundle.results]
    crashes = [bool(result.terminated) for result in bundle.results]
    summary = {
        'real_eval/avg_reward': float(sum(rewards) / len(rewards)),
        'real_eval/avg_steps': float(sum(steps) / len(steps)),
        'real_eval/crash_rate': float(sum(crashes) / len(crashes)),
        'real_eval/min_reward': float(min(rewards)),
        'real_eval/max_reward': float(max(rewards)),
    }
    return summary


def stage_should_stop(stage, demo_summary):
    too_low_reward = demo_summary['real_eval/avg_reward'] < stage['min_avg_reward']
    too_short = demo_summary['real_eval/avg_steps'] < stage['min_avg_steps']
    return too_low_reward and too_short

In [7]:
latest_checkpoint = None
stage_history = []
training_summary = None

for stage in STAGES:
    print(f"\n=== Stage: {stage['label']} | target_cycles={stage['cycles']} ===")
    training_summary = run_training_experiment(
        config,
        RUN_ARTIFACT_ROOT,
        cycles=stage['cycles'],
        warm_start_steps=config.training.warm_start_steps,
        policy_steps=config.training.policy_steps,
        checkpoint_interval=config.training.checkpoint_interval,
        resume_from=latest_checkpoint,
    )
    latest_checkpoint = training_summary.latest_checkpoint

    demo_outputs = record_demo_videos(
        config,
        checkpoint_path=latest_checkpoint,
        output_dir=RUN_ARTIFACT_ROOT / 'stage_demos' / stage['label'],
        num_episodes=stage['demo_episodes'],
        max_steps=DEMO_MAX_STEPS,
        fps=15,
        seed=config.seed,
        prefix=f"{RUN_NAME}_{stage['label']}",
        device=config.device,
    )

    demo_summary = summarize_demo(demo_outputs)
    latest_metrics = training_summary.latest_record
    stage_record = {
        'stage': stage['label'],
        'cycles': stage['cycles'],
        'checkpoint': str(latest_checkpoint),
        'world_model/total_loss': latest_metrics.get('world_model/total_loss'),
        'world_model/reward_loss': latest_metrics.get('world_model/reward_loss'),
        'behavior/actor_loss': latest_metrics.get('behavior/actor_loss'),
        'behavior/critic_loss': latest_metrics.get('behavior/critic_loss'),
        'behavior/imagined_reward_mean': latest_metrics.get('behavior/imagined_reward_mean'),
        'behavior/imagined_value_mean': latest_metrics.get('behavior/imagined_value_mean'),
        **demo_summary,
    }
    stage_history.append(stage_record)
    print(json.dumps(stage_record, indent=2))

    if AUTO_STOP and stage_should_stop(stage, demo_summary):
        print('Auto-stop triggered: real driving still looks too weak for this stage.')
        break

print('Latest checkpoint:', latest_checkpoint)


=== Stage: screen_0025 | target_cycles=25 ===
[optimizer] cast models to bfloat16 for FlashAdamW
[optimizer] using FlashAdamW
[optimizer] using FlashAdamW
[optimizer] using FlashAdamW
[train] starting run | cycles=25 | start_step=1 | warm_start_steps=1000 | policy_steps=32 | device=cuda
[train] step=1/25 | warm=1000 | policy=32 | replay=1032 | world_total=10.3186 | actor=0.1799 | critic=0.1139 | cycle_s=418.0 | elapsed_s=418.0 | checkpoint=-
[train] step=2/25 | warm=0 | policy=32 | replay=1064 | world_total=9.9103 | actor=0.3052 | critic=0.1958 | cycle_s=8.6 | elapsed_s=426.6 | checkpoint=-
[train] step=3/25 | warm=0 | policy=32 | replay=1096 | world_total=9.7105 | actor=0.4316 | critic=0.3202 | cycle_s=8.8 | elapsed_s=435.3 | checkpoint=-
[train] step=4/25 | warm=0 | policy=32 | replay=1128 | world_total=9.1334 | actor=0.6240 | critic=0.6271 | cycle_s=8.6 | elapsed_s=444.0 | checkpoint=-
[train] step=5/25 | warm=0 | policy=32 | replay=1160 | world_total=8.6885 | actor=0.9121 | critic

ValueError: replay buffer does not contain enough samples for a training cycle

In [ ]:
stage_history_path = RUN_ARTIFACT_ROOT / 'stage_history.json'
stage_history_path.write_text(json.dumps(stage_history, indent=2), encoding='utf-8')
print('Saved stage history to:', stage_history_path)
stage_history

## Optional: promote the best checkpoint to a longer run

If the stage results look good, rerun the training cell with a longer final stage or change the last stage target from `800` to `1200` or `1600`. This keeps expensive runs gated by real demo quality rather than imagination metrics alone.